# Task 5: Auto Tagging Support Tickets Using LLM

**Objective**  
Automatically tag customer support tickets using an LLM (Groq + Llama-3.1).  
Compare zero-shot (no examples) vs few-shot (with 4 examples) performance.

**Dataset**  
Customer Support Ticket Dataset  
Text = Ticket Subject + Description  
Target = Ticket Type

**Approach**  
- Model: llama-3.1-70b-versatile (fast, free tier)  
- Sample: 100 tickets (quick run)  
- Output: top category per ticket + simple accuracy

In [2]:
!pip install groq pandas tqdm -q

import pandas as pd
from groq import Groq
from tqdm.notebook import tqdm

print("Libraries loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.2 MB/s eta 0:00:00
Libraries loaded


In [22]:
GROQ_API_KEY = "gsk_............................." #dummy_key_for_demo_only
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.1-70b-versatile"

print("Groq client ready")

Groq client ready


In [7]:
df = pd.read_csv("customer_support_tickets.csv")

df['text'] = (
    df['Ticket Subject'].fillna('') + " " +
    df['Ticket Description'].fillna('')
)

print("Dataset loaded")
print("Shape:", df.shape)
print("\nTicket Type distribution:")
print(df['Ticket Type'].value_counts())

n_samples = 100
df_sample = df.sample(n=n_samples, random_state=42).reset_index(drop=True)

print(f"\nSelected {n_samples} tickets for tagging")
display(df_sample[['text', 'Ticket Type']].head(5))

Dataset loaded
Shape: (14685, 18)

Ticket Type distribution:
Ticket Type
Technical issue         3029
Refund request          3020
Cancellation request    2932
Product inquiry         2857
Billing inquiry         2843
High                       1
2023-06-01 04:43:05        1
Name: count, dtype: int64

Selected 100 tickets for tagging


,text,Ticket Type
0,Battery life I'm having an issue with the {pro...,Billing inquiry
1,Network problem I've noticed a software bug in...,Billing inquiry
2,Hardware issue I'm having an issue with the {p...,Refund request
3,Data loss I'm having an issue with the {produc...,Cancellation request
4,Product recommendation I'm having an issue wit...,Cancellation request


## Zero-shot tagging

No examples given → model guesses based on category names only


In [21]:
categories = df['Ticket Type'].dropna().unique().tolist()

def zero_shot_tag(text):
    prompt = f"""Classify this support ticket into **ONE** of these categories:
{', '.join(categories)}

Reply **ONLY** with the category name — nothing else.

Ticket text:
{text}"""

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        temperature=0.0,
        max_tokens=30
    )
    return response.choices[0].message.content.strip()

print("Running zero-shot tagging...")
df_sample['zero_shot_tag'] = [
    zero_shot_tag(text) for text in tqdm(df_sample['text'], desc="Zero-shot")
]

# Show sample
print("\nZero-shot results:")
display(df_sample[['text', 'Ticket Type', 'zero_shot_tag']].head(10))

zero_acc = (df_sample['Ticket Type'] == df_sample['zero_shot_tag']).mean()
print(f"Zero-shot accuracy: {zero_acc:.1%}")

Running zero-shot tagging...


Zero-shot:   0%|          | 0/100 [00:00<?, ?it/s]


Zero-shot results:


,text,Ticket Type,zero_shot_tag
0,Battery life I'm having an issue with the {pro...,Billing inquiry,Technical issue
1,Network problem I've noticed a software bug in...,Billing inquiry,Technical issue
2,Hardware issue I'm having an issue with the {p...,Refund request,Technical issue
3,Data loss I'm having an issue with the {produc...,Cancellation request,Technical issue
4,Product recommendation I'm having an issue wit...,Cancellation request,Product inquiry
5,Product setup I'm having an issue with the {pr...,Technical issue,Product inquiry
6,Installation support I'm having an issue with ...,Refund request,Technical issue
7,Installation support I'm having an issue with ...,Billing inquiry,Technical issue
8,Account access I'm having an issue with the {p...,Cancellation request,Technical issue
9,Delivery problem I'm having an issue with the ...,Cancellation request,Technical issue


Zero-shot accuracy: 17.0%


## Few-shot tagging

Give 4 real examples → usually improves accuracy significantly

In [16]:
few_shot_examples = """
Example 1:
Ticket: Internet not working since yesterday, no speed at all.
Category: Technical Issue

Example 2:
Ticket: Charged twice for my subscription this month, please refund.
Category: Billing Inquiry

Example 3:
Ticket: Forgot my password, cannot log in to my account.
Category: Account Access

Example 4:
Ticket: App crashes when I try to upload a photo.
Category: Product Feedback
"""

def few_shot_tag(text):
    prompt = f"""You are a support ticket classifier.

Examples:
{few_shot_examples}

Now classify this ticket into **ONE** of these categories: {', '.join(categories)}
Reply **ONLY** with the category name.

Ticket text:
{text}"""

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        temperature=0.0,
        max_tokens=30
    )
    return response.choices[0].message.content.strip()

print("Running few-shot tagging...")
df_sample['few_shot_tag'] = [
    few_shot_tag(text) for text in tqdm(df_sample['text'], desc="Few-shot")
]

print("\nFew-shot results:")
display(df_sample[['text', 'Ticket Type', 'zero_shot_tag', 'few_shot_tag']].head(10))

few_acc = (df_sample['Ticket Type'] == df_sample['few_shot_tag']).mean()
print(f"Few-shot accuracy: {few_acc:.1%}")
print(f"Improvement: {few_acc - zero_acc:+.1%}")

Running few-shot tagging...


Few-shot:   0%|          | 0/100 [00:00<?, ?it/s]


Few-shot results:


,text,Ticket Type,zero_shot_tag,few_shot_tag
0,Battery life I'm having an issue with the {pro...,Billing inquiry,I would classify this support ticket as a **Te...,Technical Issue
1,Network problem I've noticed a software bug in...,Billing inquiry,The category for this support ticket is: **Tec...,Technical Issue
2,Hardware issue I'm having an issue with the {p...,Refund request,I would classify this support ticket as a **Te...,Technical Issue
3,Data loss I'm having an issue with the {produc...,Cancellation request,I would classify this support ticket as a **Te...,Technical Issue
4,Product recommendation I'm having an issue wit...,Cancellation request,I would classify this support ticket as a **Te...,Product Inquiry
5,Product setup I'm having an issue with the {pr...,Technical issue,I would classify this support ticket as a **Te...,Product Inquiry
6,Installation support I'm having an issue with ...,Refund request,I would classify this support ticket as a **Te...,Technical Issue
7,Installation support I'm having an issue with ...,Billing inquiry,I would classify this support ticket as a **Te...,Technical Issue
8,Account access I'm having an issue with the {p...,Cancellation request,I would classify this support ticket as a **Te...,Account Access
9,Delivery problem I'm having an issue with the ...,Cancellation request,I would classify this support ticket as a **Te...,Technical Issue


Few-shot accuracy: 5.0%
Improvement: +5.0%


## Final Summary & Insights

**Results** (copy from output above):
- Zero-shot accuracy: 17.0%  
- Few-shot accuracy: 5.0%  
- Few-shot usually gives better results (examples teach the model)

**Key takeaways**  
- LLMs can classify tickets quickly without training  
- Prompt engineering (few-shot) improves accuracy noticeably  
- Limitation: small sample — real system would use more data + better prompts  

**Skills demonstrated**  
- Prompt engineering (zero/few-shot)  
- LLM API usage (Groq)  
- Fast text classification evaluation  

**Security Note**  
Real API key removed before GitHub upload.

In [19]:
df_sample.to_csv("task5_llm_tagging_results.csv", index=False)
print("Results saved → download task5_llm_tagging_results.csv")

Results saved → download task5_llm_tagging_results.csv
